# Phase A — KAN Variant Sweep

Train 5 KAN variants sequentially and compare training loss.

| Variant | Architecture |
|---------|--------------|
| **A** | B-spline + SiLU base (control) |
| **B** | B-spline + SCReLU base |
| **C** | B-spline, no base path (pure spline) |
| **D** | B-spline + linear base |
| **E** | ReLU-KAN pure basis (arXiv 2406.02075) |

Shared: `768 -> ft(128) CReLU -> KAN(256 -> 128) -> KAN(128 -> 1)`, 40 superbatches, batch 16384, AdamW, StepLR, test77 binpack.

All variants run **unfused** (the `FuseKanLayer` pass is disabled during Phase A).
Expect ~10 min / variant on T4 → ~50 min total.

**Runtime**: Runtime > Change runtime type > GPU (T4 or better).

## 1. Install Rust + clone repo

In [1]:
%%bash
if ! command -v cargo &> /dev/null; then
    curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
    echo 'source $HOME/.cargo/env' >> ~/.bashrc
fi
source $HOME/.cargo/env
rustc --version
cargo --version


  stable-x86_64-unknown-linux-gnu unchanged - rustc 1.95.0 (59807616e 2026-04-14)


Rust is installed now. Great!

To get started you may need to restart your current shell.
This would reload your PATH environment variable to include
Cargo's bin directory ($HOME/.cargo/bin).

To configure your current shell, you need to source
the corresponding env file under $HOME/.cargo.

This is usually done by running one of the following (note the leading DOT):
. "$HOME/.cargo/env"            # For sh/bash/zsh/ash/dash/pdksh
source "$HOME/.cargo/env.fish"  # For fish
source "~/.cargo/env.nu"  # For nushell
source "$HOME/.cargo/env.tcsh"  # For tcsh
. "$HOME/.cargo/env.ps1"        # For pwsh
source "$HOME/.cargo/env.xsh"   # For xonsh
rustc 1.95.0 (59807616e 2026-04-14)
cargo 1.95.0 (f2d3ce0bd 2026-03-21)


info: downloading installer
warn: It looks like you have an existing rustup settings file at:
warn: /root/.rustup/settings.toml
warn: Rustup will install the default toolchain as specified in the settings file,
warn: instead of the one inferred from the default host triple.
info: profile set to default
info: default host triple is x86_64-unknown-linux-gnu
warn: Updating existing toolchain, profile choice will be ignored
info: syncing channel updates for stable-x86_64-unknown-linux-gnu
info: default toolchain set to stable-x86_64-unknown-linux-gnu


In [2]:
%%bash
set -e
if [ -d /content/bullet ]; then
    cd /content/bullet
    git fetch origin
    git reset --hard origin/main
else
    cd /content
    git clone https://github.com/y0sif/bullet.git
    cd bullet
fi
git log -1 --oneline

HEAD is now at 151c5c0 Fix test77 download URL in phase_a_variant_sweep notebook
151c5c0 Fix test77 download URL in phase_a_variant_sweep notebook


## 2. Download training data (test77 binpack)

In [3]:
%%bash
apt-get install -y zstd 2>/dev/null || true

mkdir -p /content/bullet/data
cd /content/bullet/data

if [ ! -f test77.binpack ]; then
    echo "Downloading test77 binpack from HuggingFace (~1.3 GB compressed)..."
    wget -q -O test77.binpack.zst \
        "https://huggingface.co/datasets/linrock/test77/resolve/main/test77-2022-01-jan-2tb7p.binpack.zst"
    echo "Download complete. Decompressing..."
    zstd -d test77.binpack.zst -o test77.binpack --rm
    echo "Done!"
fi

ls -lh test77.binpack


Reading package lists...
Building dependency tree...
Reading state information...
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.
Download complete. Decompressing...
Done!
-rw-r--r-- 1 root root 2.7G Apr 19 08:59 test77.binpack


test77.binpack.zst  : 2895291738 bytes                                         


## 3. Sanity-check: GPU + nvcc

In [5]:
%%bash
nvidia-smi || echo "WARNING: No GPU detected."
echo "---"
nvcc --version || echo "WARNING: nvcc not found."


Sun Apr 19 09:00:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             16W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

bash: line 3: unexpected EOF while looking for matching `"'
bash: line 4: syntax error: unexpected end of file


CalledProcessError: Command 'b'nvidia-smi || echo "WARNING: No GPU detected."\necho "---"\nnvcc --version || echo "WARNING: nvcc not found.\n'' returned non-zero exit status 2.

## 4. Build all 5 variants (release)

The first build compiles the whole workspace + CUDA kernels (~5 min). The other four are fast.

In [ ]:
%%bash
source $HOME/.cargo/env
cd /content/bullet
for v in a b c d e; do
    echo "=== Building kan_variant_${v} ==="
    cargo build --release --example kan_variant_${v} 2>&1 | tail -3
done

In [ ]:
%%bash
ls -lh /content/bullet/data/ 2>&1
echo "---"
du -h /content/bullet/data/test77.binpack 2>&1
echo "---"
# Look at the first bytes — should start with the binpack magic, not an HTML 404
head -c 64 /content/bullet/data/test77.binpack | xxd

total 4.0K
-rw-r--r-- 1 root root 405 Apr 19 08:50 test77.binpack.zst
---
du: cannot access '/content/bullet/data/test77.binpack': No such file or directory
---


head: cannot open '/content/bullet/data/test77.binpack' for reading: No such file or directory


## 5. Train all 5 variants sequentially

Each run = 40 superbatches × ~488 batches, logs to `/content/variant_<v>_log.txt`.
If a run fails partway, earlier log files remain — you can re-run this cell starting from the failed variant by editing `VARIANTS`.

In [ ]:
import subprocess, sys, os
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]

VARIANTS = ["a", "b", "c", "d", "e"]

for v in VARIANTS:
    log_path = f"/content/variant_{v}_log.txt"
    print(f"\n{'='*70}\n Training kan_variant_{v}  ->  {log_path}\n{'='*70}")
    cmd = ["cargo", "run", "--release", "--example", f"kan_variant_{v}"]
    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        cwd="/content/bullet", text=True, bufsize=1,
    )
    with open(log_path, "w") as log:
        for line in proc.stdout:
            sys.stdout.write(line)
            sys.stdout.flush()
            log.write(line)
    proc.wait()
    print(f"\nVariant {v} exit code: {proc.returncode}")
    if proc.returncode != 0:
        print(f"FAILED — stopping sweep. Fix variant {v} before continuing.")
        break


 Training kan_variant_a  ->  /content/variant_a_log.txt
   --> crates/bullet_cuda_backend/src/ops/kan.rs:256:13
    |
256 |         let out_features = self.spline_weight.shape.rows();
    |             ^^^^^^^^^^^^ help: if this is intentional, prefix it with an underscore: `_out_features`
    |
    = note: `#[warn(unused_variables)]` (part of `#[warn(unused)]`) on by default

    Finished `release` profile [optimized] target(s) in 0.05s
     Running `target/release/examples/kan_variant_a`
Training Preamble
Net Name               : kan-variant-a
Batch Size             : 16384
Batches / Superbatch   : 488
Positions / Superbatch : 7995392
Start Superbatch       : 1
End Superbatch         : 40
Eval Scale             : 400
Save Rate              : 10
WDL Scheduler          : constant 0.75
LR Scheduler           : start 0.001 gamma 0.1 drop every 18 superbatches
Threads                : 4
Output Path            : checkpoints
Beginning Training

thread '<unnamed>' (3946) panicked at crates/

## 6. Compare loss curves + rank

In [ ]:
import re
import matplotlib.pyplot as plt

def strip_ansi(s):
    return re.sub(r'\x1b\[[0-9;]*m', '', s)

def parse_bullet_log(path):
    losses = []
    with open(path) as f:
        for line in f:
            m = re.search(r'superbatch\s+(\d+)\s+\|.*?running loss\s+([\d.]+)', strip_ansi(line))
            if m:
                losses.append((int(m.group(1)), float(m.group(2))))
    return losses

VARIANT_LABELS = {
    "a": "A: B-spline + SiLU (control)",
    "b": "B: B-spline + SCReLU",
    "c": "C: B-spline (no base)",
    "d": "D: B-spline + linear",
    "e": "E: ReLU-KAN",
}

all_losses = {}
for v in ["a", "b", "c", "d", "e"]:
    path = f"/content/variant_{v}_log.txt"
    try:
        losses = parse_bullet_log(path)
    except FileNotFoundError:
        print(f"Skipping variant {v}: no log at {path}")
        continue
    if losses:
        all_losses[v] = losses
    else:
        print(f"Variant {v}: log exists but no superbatch lines parsed")

if all_losses:
    fig, ax = plt.subplots(figsize=(11, 7))
    for v, losses in all_losses.items():
        ax.plot([x[0] for x in losses], [x[1] for x in losses],
                label=VARIANT_LABELS[v], linewidth=2)
    ax.set_xlabel("Superbatch")
    ax.set_ylabel("Running loss")
    ax.set_title("Phase A — KAN Variant Sweep (40 superbatches each)")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("/content/phase_a_sweep.png", dpi=150)
    plt.show()

    print("\nFinal loss (lower = better):")
    ranked = sorted(all_losses.items(), key=lambda kv: kv[1][-1][1])
    for rank, (v, losses) in enumerate(ranked, 1):
        marker = " <-- best" if rank == 1 else ""
        print(f"  {rank}. {VARIANT_LABELS[v]:<38}  {losses[-1][1]:.6f}{marker}")
else:
    print("No variant logs parsed.")

## 7. Save results (optional)

Copy logs + plot to Drive for later analysis.

In [ ]:
import shutil, os
from google.colab import drive
drive.mount('/content/drive')

dest = '/content/drive/MyDrive/kanue/phase_a'
os.makedirs(dest, exist_ok=True)

for v in ["a", "b", "c", "d", "e"]:
    src = f"/content/variant_{v}_log.txt"
    if os.path.exists(src):
        shutil.copy(src, dest)

if os.path.exists('/content/phase_a_sweep.png'):
    shutil.copy('/content/phase_a_sweep.png', dest)

# Save checkpoints if any variant produced them
ckpt_root = '/content/bullet/checkpoints'
if os.path.isdir(ckpt_root):
    for d in os.listdir(ckpt_root):
        if d.startswith('kan-variant-'):
            src = os.path.join(ckpt_root, d)
            shutil.copytree(src, os.path.join(dest, 'checkpoints', d), dirs_exist_ok=True)

print(f"Saved to: {dest}")